In [1]:
import sys
if 'google.colab' in sys.modules:
  !pip install pyrosettacolabsetup
  import pyrosettacolabsetup
  pyrosettacolabsetup.mount_pyrosetta_install()
from pyrosetta import *
init()


Mounted at /content/google_drive

Note that USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRE PURCHASE OF A LICENSE.
See https://github.com/RosettaCommons/rosetta/blob/main/LICENSE.md or email license@uw.edu for details.

Looking for compatible PyRosetta wheel file at google-drive/PyRosetta/colab.bin//wheels...
--2026-07-23 00:04:41--  https://west.rosettacommons.org/pyrosetta/quarterly/release/pyrosetta-0-cp312-cp312-linux_x86_64.whl

Connecting to west.rosettacommons.org (west.rosettacommons.org)|128.95.160.135|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1662376546 (1.5G) [application/octet-stream]
--2026-07-23 00:04:41--  https://west.rosettacommons.org/pyrosetta/quarterly/release/pyrosetta-0-cp312-cp312-linux_x86_64.whl

HTTP request sent, awaiting response... 200 OK
Length: 1662376546 (1.5G) [application/octet-stream]
Saving to: ‘/content/google_drive/MyDrive/PyRosetta/colab.bin/tmpj3540xmh/pyrosetta-0-cp312-cp312-linux_x86_64.whl’

pyrosetta-0-cp312-c

##Get a PDB
Go to the REU 2026 Bootcamp Website and download the PDB
This PDB is a file of a de-novo designed binder bound to the Sirtuin-1 protein.


P53 is a tumor suppressor protein and SIRT-1 has been thought to downregulate P53 in response to DNA damage or cell stress, inhibiting P53's tumor suppressor functionality. It's tumor suppressor functionality extends to cell apoptosis (getting rid of damaged DNA), sensence, and plays a role in the cell cycle to help get rid of mutated DNA. Blocking this interaction between the SIRT1 protein and P53 with a binder could help inhibit P53's downregulation.


We're going to analyze the interface of this interaction between SIRT-1 and the de-novo binder


Upload the pdb file into the next upload cell

In [2]:
from google.colab import files

uploaded = files.upload()

pdb_file = next(iter(uploaded))
print(f"Uploaded: {pdb_file}")

Saving r4_i6_0_85_seq_1049_seq_911_unrelaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000_relaxed.pdb to r4_i6_0_85_seq_1049_seq_911_unrelaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000_relaxed.pdb
Uploaded: r4_i6_0_85_seq_1049_seq_911_unrelaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000_relaxed.pdb


##Load your PDB File

In [3]:
from pyrosetta import *

pose = pose_from_pdb(pdb_file)

print("Loaded:", pdb_file)
print("Residues:", pose.total_residue())

core.chemical.GlobalResidueTypeSet: Finished initializing fa_standard residue type set.  Created 985 residue types
core.chemical.GlobalResidueTypeSet: Total time to initialize 1.59376 seconds.
core.import_pose.import_pose: File 'r4_i6_0_85_seq_1049_seq_911_unrelaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000_relaxed.pdb' automatically determined to be of type PDB from contents.
Loaded: r4_i6_0_85_seq_1049_seq_911_unrelaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000_relaxed.pdb
Residues: 416


In [4]:
# @title
import ipywidgets as widgets
from IPython.display import display, Markdown

display(Markdown("## Checkpoint: Interface Analyzer Questions"))

questions = {
    "Interface Analyzer Questions": [
        "Knowing that this binder was designed to bind to SIRT1, how do you think the interface analyzer will score the delta of the interface?",
        "How do you think the interface analyzer will score the delta G of the separated chains?",
        "How do you think the SASA will be affected when bound vs unboud to each other?",
    ],
}
answer_boxes = {}

for section, qs in questions.items():

    display(Markdown(f"### {section}"))

    for q in qs:

        display(Markdown(f"**{q}**"))

        box = widgets.Textarea(
            placeholder="Enter your answer here...",
            layout=widgets.Layout(width='90%', height='20%')
        )

        answer_boxes[q] = box
        display(box)

## Checkpoint: Interface Analyzer Questions

### Interface Analyzer Questions

**Knowing that this binder was designed to bind to SIRT1, how do you think the interface analyzer will score the delta of the interface?**

Textarea(value='', layout=Layout(height='20%', width='90%'), placeholder='Enter your answer here...')

**How do you think the interface analyzer will score the delta G of the separated chains?**

Textarea(value='', layout=Layout(height='20%', width='90%'), placeholder='Enter your answer here...')

**How do you think the SASA will be affected when bound vs unboud to each other?**

Textarea(value='', layout=Layout(height='20%', width='90%'), placeholder='Enter your answer here...')

In [5]:
interface = "A_B"

##Run Rosetta Interface Analyzer

In [6]:
# @title
import pandas as pd
from pyrosetta.rosetta.protocols.analysis import InterfaceAnalyzerMover
from pyrosetta.rosetta.protocols.simple_filters import ShapeComplementarityFilter
from pyrosetta.rosetta.protocols.simple_filters import BuriedUnsatHbondFilter

sc = ShapeComplementarityFilter()

print("SC:", sc.compute(pose))


buns = BuriedUnsatHbondFilter()

print(buns.compute(pose))
iam = InterfaceAnalyzerMover(interface)
iam.set_compute_packstat(True)

print("Running InterfaceAnalyzer...")
iam.apply(pose)

for m in dir(iam):
    if "hbond" in m.lower():
        print(m)

results = {}

for method_name in dir(iam):

    if not method_name.startswith("get_"):
        continue

    try:
        method = getattr(iam, method_name)

        if callable(method):
            value = method()

            if isinstance(value, (int, float, str, bool)):
                results[method_name] = value

    except:
        pass

df = pd.DataFrame(
    list(results.items()),
    columns=["Metric", "Value"]
)
df = df[
    ~df["Metric"].isin([
        "get_pymol_sel_hbond_unsat",
        "get_pymol_sel_interface",
        "get_pymol_sel_packing",
    ])
]

display(df)

# ---------------------------------
# Save CSV named after PDB
# ---------------------------------

csv_name = pdb_file.replace(".pdb", "_rosetta_metrics.csv")

df.to_csv(csv_name, index=False)

print(f"\nSaved: {csv_name}")


basic.io.database: Database file opened: scoring/score_functions/sc/sc_radii.lib
SC: <pyrosetta.rosetta.core.scoring.sc._RESULTS object at 0x7dc1c44e84f0>
protocols.simple_filters.BuriedUnsatHbondFilter: /////////////////////////////////////////////////////////////////////////////////////////
protocols.simple_filters.BuriedUnsatHbondFilter:
protocols.simple_filters.BuriedUnsatHbondFilter:     USING DEFAULT BEHAVIOR: filter will report total number of heavy-atom donor/acceptor buried unsats
protocols.simple_filters.BuriedUnsatHbondFilter:
protocols.simple_filters.BuriedUnsatHbondFilter: /////////////////////////////////////////////////////////////////////////////////////////
protocols.simple_filters.BuriedUnsatHbondFilter:
core.scoring.ScoreFunctionFactory: SCOREFUNCTION: ref2015
core.scoring.etable: Starting energy table calculation
core.scoring.etable: smooth_etable: changing atr/rep split to bottom of energy well
core.scoring.etable: smooth_etable: spline smoothing lj etables (maxdis

,Metric,Value
0,get_centroid_dG,0.0
1,get_complex_energy,-914.572731
2,get_complexed_sasa,20981.078125
3,get_crossterm_interface_energy,-24.960541
4,get_crossterm_interface_energy_ratio,-0.013806
5,get_current_tag,NoTag
6,get_gly_interface_energy,0.0
7,get_interface_dG,-24.960541
8,get_interface_delta_hbond_unsat,7
9,get_interface_delta_sasa,1807.967622



Saved: r4_i6_0_85_seq_1049_seq_911_unrelaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000_relaxed_rosetta_metrics.csv


In [7]:
# @title
import ipywidgets as widgets
from IPython.display import display, Markdown

display(Markdown("## Checkpoint: Interface Analyzer Questions"))

questions = {
    "Interface Analyzer Questions": [
        "How many interface residues does your protein complex have?",
        "What is the complexed energy? Is this a strong complexed energy?",
        "What is the number of residues on side 1 vs side 2?",
        "What is your per residue energy?",

    ],
}
answer_boxes = {}

for section, qs in questions.items():

    display(Markdown(f"### {section}"))

    for q in qs:

        display(Markdown(f"**{q}**"))

        box = widgets.Textarea(
            placeholder="Enter your answer here...",
            layout=widgets.Layout(width='90%', height='20%')
        )

        answer_boxes[q] = box
        display(box)

## Checkpoint: Interface Analyzer Questions

### Interface Analyzer Questions

**How many interface residues does your protein complex have?**

Textarea(value='', layout=Layout(height='20%', width='90%'), placeholder='Enter your answer here...')

**What is the complexed energy? Is this a strong complexed energy?**

Textarea(value='', layout=Layout(height='20%', width='90%'), placeholder='Enter your answer here...')

**What is the number of residues on side 1 vs side 2?**

Textarea(value='', layout=Layout(height='20%', width='90%'), placeholder='Enter your answer here...')

**What is your per residue energy?**

Textarea(value='', layout=Layout(height='20%', width='90%'), placeholder='Enter your answer here...')

In [8]:
# @title
import pandas as pd

from pyrosetta.rosetta.protocols.analysis import InterfaceAnalyzerMover
from pyrosetta.rosetta.protocols.simple_filters import ShapeComplementarityFilter
from pyrosetta.rosetta.core.scoring import get_score_function
from pyrosetta.rosetta.core.scoring.sasa import SasaCalc

# -----------------------------
# Total score
# -----------------------------
scorefxn = get_score_function()

total_score = scorefxn(pose)

# -----------------------------
# Interface analysis
# -----------------------------
iam = InterfaceAnalyzerMover(interface)

try:
    iam.set_compute_interface_delta_hbond_unsat(True)
except:
    pass

iam.set_compute_packstat(True)
iam.apply(pose)

# -----------------------------
# Shape Complementarity
# -----------------------------
sc_filter = ShapeComplementarityFilter()

try:
    sc_value = sc_filter.compute(pose)
except:
    sc_value = None

# -----------------------------
# Total SASA
# -----------------------------
sasa_calc = SasaCalc()
sasa_calc.calculate(pose)

total_sasa = sasa_calc.get_total_sasa()

# -----------------------------
# Collect results
# -----------------------------
results = {
    "score": total_score,
    "dG_separated": iam.get_interface_dG(),
    "dSASA_int": iam.get_interface_delta_sasa(),
    "delta_unsatHbonds": iam.get_interface_delta_hbond_unsat(),
    "sc": sc_value,
    "sasa": total_sasa,
}

df_small = pd.DataFrame(
    list(results.items()),
    columns=["Metric", "Value"]
)

display(df_small)
# ---------------------------------
# Save CSV named after PDB
# ---------------------------------

csv_small_name = pdb_file.replace(".pdb", "_rosetta_metrics_small.csv")

df_small.to_csv(csv_name, index=False)

print(f"\nSaved: {csv_small_name}")

core.scoring.ScoreFunctionFactory: SCOREFUNCTION: ref2015
protocols.analysis.InterfaceAnalyzerMover: Using explicit constructor
protocols.analysis.InterfaceAnalyzerMover: Using interface constructor
protocols.analysis.InterfaceAnalyzerMover: [ WARNING ] In InterfaceAnalyzerMover, calculator Sasa_1 already exists, this is hopefully correct for your purposes
protocols.analysis.InterfaceAnalyzerMover: [ WARNING ] In InterfaceAnalyzerMover, calculator InterfaceNeighborDefinition_1 already exists, this is hopefully correct for your purposes
protocols.analysis.InterfaceAnalyzerMover: [ WARNING ] In InterfaceAnalyzerMover, calculator InterfaceDeltaEnergetics_1 already exists, this is hopefully correct for your purposes
protocols.analysis.InterfaceAnalyzerMover: [ WARNING ] In InterfaceAnalyzerMover, calculator NumberHBonds_1 already exists, this is hopefully correct for your purposes
protocols.analysis.InterfaceAnalyzerMover: [ WARNING ] In InterfaceAnalyzerMover, calculator BuriedUnsatisfied

,Metric,Value
0,score,-914.572731
1,dG_separated,-24.960541
2,dSASA_int,1807.967622
3,delta_unsatHbonds,7
4,sc,<pyrosetta.rosetta.core.scoring.sc._RESULTS ob...
5,sasa,20981.078125



Saved: r4_i6_0_85_seq_1049_seq_911_unrelaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000_relaxed_rosetta_metrics_small.csv


In [9]:
# @title
import ipywidgets as widgets
from IPython.display import display, Markdown

display(Markdown("## Checkpoint:Interface Questions continued"))

questions = {
    "Interface Analyzer Questions continued": [
        "What is your score?",
        "What is dSASA_int? What does this say about the SASA when the complex is bound vs when not bound?",
        "What is dG_separated? What does this say about the dG when the complex is bound vs when not bound?",
        "What is your SASA?"
    ],
}
answer_boxes = {}

for section, qs in questions.items():

    display(Markdown(f"### {section}"))

    for q in qs:

        display(Markdown(f"**{q}**"))

        box = widgets.Textarea(
            placeholder="Enter your answer here...",
            layout=widgets.Layout(width='90%', height='20%')
        )

        answer_boxes[q] = box
        display(box)

## Checkpoint:Interface Questions continued

### Interface Analyzer Questions continued

**What is your score?**

Textarea(value='', layout=Layout(height='20%', width='90%'), placeholder='Enter your answer here...')

**What is dSASA_int? What does this say about the SASA when the complex is bound vs when not bound?**

Textarea(value='', layout=Layout(height='20%', width='90%'), placeholder='Enter your answer here...')

**What is dG_separated? What does this say about the dG when the complex is bound vs when not bound?**

Textarea(value='', layout=Layout(height='20%', width='90%'), placeholder='Enter your answer here...')

**What is your SASA?**

Textarea(value='', layout=Layout(height='20%', width='90%'), placeholder='Enter your answer here...')

##Optional: Relaxing (~5 mins)
Relaxing is a Rosetta function that will take your protein structure and try to minimize the energy of your protein by slightly moving side chains to get closer to an energy minima

In [10]:
# @title
import ipywidgets as widgets
from IPython.display import display, Markdown

display(Markdown("## Checkpoint: Relaxing Questions"))

questions = {
    "Relaxing question": [
        "Do you think that relaxing the structure will minimize you Rosetta Ref2015 score? Why?",

    ],
}
answer_boxes = {}

for section, qs in questions.items():

    display(Markdown(f"### {section}"))

    for q in qs:

        display(Markdown(f"**{q}**"))

        box = widgets.Textarea(
            placeholder="Enter your answer here...",
            layout=widgets.Layout(width='90%', height='20%')
        )

        answer_boxes[q] = box
        display(box)

## Checkpoint: Relaxing Questions

### Relaxing question

**Do you think that relaxing the structure will minimize you Rosetta Ref2015 score? Why?**

Textarea(value='', layout=Layout(height='20%', width='90%'), placeholder='Enter your answer here...')

In [11]:
# @title
from pyrosetta import *
from pyrosetta.rosetta.protocols.relax import FastRelax
from pyrosetta.rosetta.core.scoring import get_score_function

# -----------------------------
# Load original structure
# -----------------------------

original_pose = pose_from_pdb(pdb_file)

# Make a copy for relaxation
relaxed_pose = Pose()
relaxed_pose.assign(original_pose)

# -----------------------------
# Score function
# -----------------------------

scorefxn = get_score_function()

# Score before relaxation
score_before = scorefxn(original_pose)

print("Initial score:")
print(score_before)

# -----------------------------
# FastRelax
# -----------------------------

relax = FastRelax()

relax.set_scorefxn(scorefxn)

# Fewer repeats for Colab speed
relax.max_iter(200)

print("\nRunning FastRelax...")

relax.apply(relaxed_pose)

# -----------------------------
# Score after relaxation
# -----------------------------

score_after = scorefxn(relaxed_pose)

print("\nRelax complete")

print(f"\nScore before: {score_before:.3f}")
print(f"Score after : {score_after:.3f}")
print(f"Improvement : {score_before - score_after:.3f}")

# -----------------------------
# Save relaxed PDB
# -----------------------------

relaxed_pdb = pdb_file.replace(".pdb", "_relaxed.pdb")

relaxed_pose.dump_pdb(relaxed_pdb)

print(f"\nSaved: {relaxed_pdb}")

core.import_pose.import_pose: File 'r4_i6_0_85_seq_1049_seq_911_unrelaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000_relaxed.pdb' automatically determined to be of type PDB from contents.
core.scoring.ScoreFunctionFactory: SCOREFUNCTION: ref2015
Initial score:
-914.5727314502806
protocols.relax.RelaxScriptManager: Reading relax scripts list from database.
core.scoring.ScoreFunctionFactory: SCOREFUNCTION: ref2015
protocols.relax.RelaxScriptManager: Looking for MonomerRelax2019.txt
protocols.relax.RelaxScriptManager: ================== Reading script file: /usr/local/lib/python3.12/dist-packages/pyrosetta/database/sampling/relax_scripts/MonomerRelax2019.txt ==================
protocols.relax.RelaxScriptManager: repeat %%nrepeats%%
protocols.relax.RelaxScriptManager: coord_cst_weight 1.0
protocols.relax.RelaxScriptManager: scale:fa_rep 0.040
protocols.relax.RelaxScriptManager: repack
protocols.relax.RelaxScriptManager: scale:fa_rep 0.051
protocols.relax.RelaxScriptManager: min 0.01


In [12]:
# @title
import ipywidgets as widgets
from IPython.display import display, Markdown

display(Markdown("## Checkpoint: Relaxing Questions"))

questions = {
    "Relaxing question": [
        "Did relaxing the structure minimize the score?",
        "How much did the score get minimized if at all?",

    ],
}
answer_boxes = {}

for section, qs in questions.items():

    display(Markdown(f"### {section}"))

    for q in qs:

        display(Markdown(f"**{q}**"))

        box = widgets.Textarea(
            placeholder="Enter your answer here...",
            layout=widgets.Layout(width='90%', height='20%')
        )

        answer_boxes[q] = box
        display(box)

## Checkpoint: Relaxing Questions

### Relaxing question

**Did relaxing the structure minimize the score?**

Textarea(value='', layout=Layout(height='20%', width='90%'), placeholder='Enter your answer here...')

**How much did the score get minimized if at all?**

Textarea(value='', layout=Layout(height='20%', width='90%'), placeholder='Enter your answer here...')